# NCex E5 summary tables from a wandb export

Builds the two E5 tables from a wandb runs export (project `NCex_SAIL_revision`, **Runs → Export → CSV**):

1. **Every model learned the shortcut**: AUROC on T1 / T2 / T3 and cheat reliance = AUROC T1 − AUROC T3.
2. **Do the heatmaps show the marker? (on marked images, T2)**, per model and explainer:
   - *Marker relevance* = heatmap mass inside the marker box ÷ total heatmap mass (disease-class maps); chance = box area ÷ frame area.
   - *Pointing hit rate* = how often the hottest pixel falls inside the marker box (box widened by 2 pixels).

Values come from the per-model eval runs (`NCex_E5-eval-<model>`); the merge run is not needed.
See `docs/plans/2026-10-13-ncex-e4-e5-design.md`.

In [ ]:
# ---- settings
CSV_PATH = 'wandb_export.csv'  # the wandb runs export
RUN_ID = None                  # e.g. 'E5_v2' or 'E5_v2_bg'; required if the export holds several E5 runs
OUT_DIR = None                 # e.g. 'tables/' to also save table1/table2 as .csv and tables.md
KEEP_IDENTICAL = False         # True: one row per explainer even when a model's explainers give identical values

In [ ]:
import os

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

EXPLAINERS = {'gradcamv2': 'Grad-CAM', 'gradcam++': 'Grad-CAM++', 'hirescam': 'HiResCAM'}
EVAL_PREFIX = 'NCex_E5-eval-'


def load_eval_runs(path, run_id):
    df = pd.read_csv(path)
    runs = df[df['Name'].astype(str).str.startswith(EVAL_PREFIX)].copy()
    if 'State' in runs.columns:
        runs = runs[runs['State'] == 'finished']
    if run_id is not None:
        runs = runs[runs['run_id'].astype(str) == run_id]
    elif 'run_id' in runs.columns and runs['run_id'].nunique() > 1:
        raise ValueError(f'Export holds several run ids {sorted(runs["run_id"].unique())}; set RUN_ID')
    if runs.empty:
        raise ValueError(f'No finished {EVAL_PREFIX}* runs found' + (f' for run_id {run_id}' if run_id else ''))
    runs['model'] = runs['model_tag'] if 'model_tag' in runs.columns else runs['Name'].str[len(EVAL_PREFIX):]
    if 'Created' in runs.columns:  # a re-run of the same model: keep the latest
        runs = runs.sort_values('Created')
    dups = runs[runs.duplicated('model', keep='last')]['model'].unique()
    if len(dups):
        print(f'[note] several finished eval runs for {list(dups)}; using the latest of each')
    return runs.drop_duplicates('model', keep='last').set_index('model')


def table1(runs):
    t = pd.DataFrame({'AUROC T1 (clean)': runs['T1/auroc'], 'AUROC T2 (all marked)': runs['T2/auroc'],
                      'AUROC T3 (healthy marked)': runs['T3/auroc']})
    t['Cheat reliance'] = t['AUROC T1 (clean)'] - t['AUROC T3 (healthy marked)']
    return t.sort_values('Cheat reliance', ascending=False)


def table2(runs, order, keep_identical):
    rows = []
    for model in order:
        r = runs.loc[model]
        per = []
        for key, name in EXPLAINERS.items():
            col = f'T2/{key}/marker_mass_mean'
            if col not in r.index or pd.isna(r[col]):
                continue
            per.append({'Model': model, 'Explainer': name,
                        'Marker relevance': float(r[col]),
                        'Chance': float(r[f'T2/{key}/chance']),
                        'Pointing hit rate': float(r[f'T2/{key}/pointing_hit_rate']),
                        'All-zero maps': int(r.get(f'T2/{key}/degenerate', 0) or 0),
                        'n': int(r.get('T2/n', 0) or 0)})
        values = np.array([[p['Marker relevance'], p['Pointing hit rate']] for p in per])
        if not keep_identical and len(per) > 1 and np.allclose(values, values[0], rtol=1e-6, atol=1e-12):
            per = [dict(per[0], Explainer=f'all {len(per)} (identical)')]
        rows += per
    return pd.DataFrame(rows)


def pct(x):
    """Percent with enough digits for very small values (e.g. 0.0006%)."""
    if pd.isna(x):
        return 'n/a'
    v = 100 * x
    if v == 0:
        return '0.0%'
    return f'{v:.4f}%' if v < 0.01 else f'{v:.2f}%'


def hit(x):
    if pd.isna(x):
        return 'n/a'
    v = 100 * x
    return f'{v:.2f}%' if 0 < v < 1 else f'{v:.1f}%'


def markdown(df):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    lines += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in df.itertuples(index=False)]
    return '\n'.join(lines)

In [ ]:
runs = load_eval_runs(CSV_PATH, RUN_ID)
t1 = table1(runs)
t2 = table2(runs, list(t1.index), KEEP_IDENTICAL)
run_id = RUN_ID or (runs['run_id'].iloc[0] if 'run_id' in runs.columns else '')
n = int(runs['T2/n'].iloc[0]) if 'T2/n' in runs.columns else None
print(f'E5 {run_id}' + (f' ({n} test images)' if n else '') + f'; models: {list(t1.index)}')

## 1. Every model learned the shortcut

In [ ]:
f1 = t1.reset_index().rename(columns={'model': 'Model'})
for c in f1.columns[1:]:
    f1[c] = f1[c].map(lambda v: f'{v:.3f}')
display(Markdown(markdown(f1) + '\n\nCheat reliance = AUROC on T1 minus AUROC on T3. '
                 'A large drop means the model relies on the marker.'))

## 2. Do the heatmaps show the marker? (on marked images, T2)

In [ ]:
f2 = pd.DataFrame({'Model': t2['Model'], 'Explainer': t2['Explainer'],
                   'Marker relevance (chance)': [f'{pct(m)} ({pct(c)})' for m, c in zip(t2['Marker relevance'], t2['Chance'])],
                   'Pointing hit rate': t2['Pointing hit rate'].map(hit)})
display(Markdown(markdown(f2) + '\n\nMarker relevance = heatmap mass inside the marker box / total heatmap mass '
                 '(disease-class maps); chance = box area / frame area. Pointing hit rate = how often the hottest pixel '
                 'falls inside the marker box (box widened by 2 pixels).'))

All-zero heatmaps are left out of the values above:

In [ ]:
zero = t2[t2['All-zero maps'] > 0][['Model', 'Explainer', 'All-zero maps', 'n']]
zero = zero.assign(share=(100 * zero['All-zero maps'] / zero['n'].clip(lower=1)).round(0).astype(int).astype(str) + '%')
zero if len(zero) else print('none')

## Save (optional)

In [ ]:
if OUT_DIR:
    os.makedirs(OUT_DIR, exist_ok=True)
    t1.reset_index().rename(columns={'model': 'Model'}).to_csv(os.path.join(OUT_DIR, 'table1_shortcut.csv'), index=False)
    t2.to_csv(os.path.join(OUT_DIR, 'table2_heatmaps.csv'), index=False)
    with open(os.path.join(OUT_DIR, 'tables.md'), 'w', encoding='utf-8') as f:
        f.write(markdown(f1) + '\n\n' + markdown(f2) + '\n')
    print(f'Written to {OUT_DIR}: table1_shortcut.csv, table2_heatmaps.csv, tables.md')